# Waste AI — Meilleur modèle (v3)
**EfficientNet-B2 + TrashNet + TACO (téléchargement robuste) + Augmentation maximale**

Objectif : modèle le plus polyvalent possible — fonctionne sur fond blanc, fond complexe,
mauvaise lumière, plusieurs objets, photos de téléphone.

**Exécute les cellules dans l'ordre. GPU T4 obligatoire.**

## Étape 1 — Installation & vérification GPU

In [ ]:
!pip install torch torchvision timm albumentations scikit-learn seaborn matplotlib requests tqdm -q

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE != 'cuda':
    raise RuntimeError('GPU non actif ! Va dans Exécution > Modifier le type > GPU T4')
print('GPU OK — on peut commencer')

## Étape 2 — Téléchargement TrashNet

In [ ]:
import os

if not os.path.exists('trashnet/dataset-resized'):
    print('Téléchargement TrashNet...')
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
    !unzip -q dataset-resized.zip -d trashnet/
    print('TrashNet OK')
else:
    print('TrashNet déjà présent')

classes = sorted(os.listdir('trashnet/dataset-resized'))
print('Classes :', classes)
for c in classes:
    n = len(os.listdir(f'trashnet/dataset-resized/{c}'))
    print(f'  {c}: {n} images')

## Étape 3 — Téléchargement TACO (méthode robuste via URLs directes)

In [ ]:
import os, json, requests
from tqdm import tqdm

# Cloner le repo pour récupérer annotations.json
if not os.path.exists('TACO'):
    !git clone --quiet https://github.com/pedropro/TACO.git
    print('Repo TACO cloné')
else:
    print('Repo TACO déjà présent')

TACO_IMG_DIR = 'TACO/data/images'
os.makedirs(TACO_IMG_DIR, exist_ok=True)

with open('TACO/data/annotations.json') as f:
    taco_data = json.load(f)

print(f'{len(taco_data["images"])} images référencées dans TACO')

# Télécharger les images directement depuis leur URL
errors = 0
for img in tqdm(taco_data['images'], desc='Téléchargement TACO'):
    dst = os.path.join(TACO_IMG_DIR, f"{img['id']}.jpg")
    if os.path.exists(dst):
        continue
    try:
        r = requests.get(img['flickr_url'], timeout=15)
        if r.status_code == 200:
            with open(dst, 'wb') as f:
                f.write(r.content)
        else:
            # Essayer l'URL de secours
            r2 = requests.get(img['coco_url'], timeout=15)
            if r2.status_code == 200:
                with open(dst, 'wb') as f:
                    f.write(r2.content)
            else:
                errors += 1
    except Exception:
        errors += 1

downloaded = len(os.listdir(TACO_IMG_DIR))
print(f'\nTACO : {downloaded} images téléchargées ({errors} erreurs)')

## Étape 4 — Organisation TACO par classe

In [ ]:
import os, json, shutil
from PIL import Image

TACO_TO_CLASS = {
    'Plastic bottle': 'plastic', 'Plastic bag & wrapper': 'plastic',
    'Plastic container': 'plastic', 'Plastic cup': 'plastic',
    'Plastic lid': 'plastic', 'Plastic straw': 'plastic',
    'Plastic utensils': 'plastic', 'Six pack rings': 'plastic',
    'Styrofoam piece': 'plastic',
    'Paper': 'paper', 'Paper bag': 'paper', 'Newspaper': 'paper',
    'Cardboard': 'cardboard', 'Carton': 'cardboard', 'Drink carton': 'cardboard',
    'Glass bottle': 'glass', 'Broken glass': 'glass', 'Glass jar': 'glass',
    'Aluminium foil': 'metal', 'Metal bottle cap': 'metal',
    'Drink can': 'metal', 'Food can': 'metal', 'Pop tab': 'metal',
    'Scrap metal': 'metal',
    'Cigarette': 'trash', 'Rope & strings': 'trash',
    'Shoe': 'trash', 'Unlabeled litter': 'trash',
}

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
TACO_OUT = 'taco_organized'
TACO_IMG_DIR = 'TACO/data/images'

for cls in CLASS_NAMES:
    os.makedirs(f'{TACO_OUT}/{cls}', exist_ok=True)

with open('TACO/data/annotations.json') as f:
    taco_data = json.load(f)

cat_id_to_name = {cat['id']: cat['name'] for cat in taco_data['categories']}

count = 0
for ann in taco_data['annotations']:
    cat_name = cat_id_to_name.get(ann['category_id'], '')
    target_class = TACO_TO_CLASS.get(cat_name)
    if target_class is None:
        continue
    src = os.path.join(TACO_IMG_DIR, f"{ann['image_id']}.jpg")
    dst = f"{TACO_OUT}/{target_class}/{ann['image_id']}_{ann['id']}.jpg"
    if os.path.exists(src) and not os.path.exists(dst):
        # Vérifier que l'image est valide
        try:
            Image.open(src).verify()
            shutil.copy(src, dst)
            count += 1
        except Exception:
            pass

print(f'TACO : {count} images organisées')
for cls in CLASS_NAMES:
    n = len(os.listdir(f'{TACO_OUT}/{cls}'))
    print(f'  {cls}: {n} images')

## Étape 5 — Dataset avec augmentation maximale

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from torchvision import datasets
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
TACO_OUT = 'taco_organized'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class WasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.data = datasets.ImageFolder(root)
        self.transform = transform
        self.classes = self.data.classes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data.samples[idx]
        image = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label


# Augmentation maximale — simule toutes les conditions réelles
train_transform = A.Compose([
    A.RandomResizedCrop(size=(260, 260), scale=(0.4, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=90, p=0.5),
    # Simule fond complexe
    A.OneOf([
        A.GaussNoise(std_range=(0.05, 0.2)),
        A.GaussianBlur(blur_limit=5),
        A.MotionBlur(blur_limit=5),
        A.ImageCompression(quality_range=(30, 80)),
    ], p=0.5),
    # Simule conditions lumière variées
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4),
        A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=30),
        A.CLAHE(clip_limit=6),
        A.RandomShadow(p=1.0),
    ], p=0.6),
    # Simule occlusion partielle
    A.CoarseDropout(num_holes_range=(1, 12), hole_height_range=(8, 48), hole_width_range=(8, 48), p=0.4),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=260, width=260),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

trashnet_ds = WasteDataset('trashnet/dataset-resized', transform=train_transform)
taco_ds = WasteDataset(TACO_OUT, transform=train_transform)

print(f'TrashNet : {len(trashnet_ds)} images')
print(f'TACO     : {len(taco_ds)} images')

full_dataset = ConcatDataset([trashnet_ds, taco_ds])
print(f'Total    : {len(full_dataset)} images')

n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train : {n_train} | Val : {n_val}')

## Étape 6 — Modèle EfficientNet-B2 (bien plus puissant que MobileNetV3)

In [ ]:
import torch
import torch.nn as nn
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)

# EfficientNet-B2 : meilleur compromis précision / vitesse
# Bien meilleur que MobileNetV3 sur des images réelles complexes
model = timm.create_model('efficientnet_b2', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'EfficientNet-B2 prêt sur {DEVICE}')
print(f'Paramètres : {total_params:.1f}M')
print(f'Classes : {CLASS_NAMES}')

## Étape 7 — Entraînement en 2 phases

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
history = {'train_loss': [], 'val_acc': []}


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total * 100


# Phase 1 — geler le backbone, entraîner la tête (5 epochs)
print('Phase 1 : tête uniquement (5 epochs)')
for name, param in model.named_parameters():
    param.requires_grad = 'classifier' in name

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

for epoch in range(5):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f'  Epoch {epoch+1}/5 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%')


# Phase 2 — fine-tuning complet avec lr très faible (20 epochs)
print('\nPhase 2 : fine-tuning complet (20 epochs)')
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {'params': model.conv_stem.parameters(), 'lr': 1e-5},
    {'params': model.blocks.parameters(), 'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

best_acc = 0
for epoch in range(20):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f'  Epoch {epoch+1}/20 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%')
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'checkpoints/waste_ai_v3.pt')
        print(f'    -> Meilleur modèle sauvegardé ({acc:.1f}%)')

print(f'\nMeilleure accuracy finale : {best_acc:.1f}%')

## Étape 8 — Évaluation complète

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import torch

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Charger le meilleur modèle
model.load_state_dict(torch.load('checkpoints/waste_ai_v3.pt'))
model.eval()

# Courbes d'entraînement
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color='#e74c3c')
ax1.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax2.plot(history['val_acc'], color='#2ecc71')
ax2.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax2.set_title('Val Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.legend()
plt.suptitle('Waste AI v3 — EfficientNet-B2 + TrashNet + TACO', fontsize=13)
plt.tight_layout()
plt.savefig('checkpoints/training_curves_v3.png', dpi=150)
plt.show()

# Matrice de confusion
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print('\n=== Rapport de classification ===')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Matrice de confusion — Waste AI v3')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('checkpoints/confusion_matrix_v3.png', dpi=150)
plt.show()

## Étape 9 — Télécharger le modèle

In [ ]:
from google.colab import files

files.download('checkpoints/waste_ai_v3.pt')
files.download('checkpoints/confusion_matrix_v3.png')
files.download('checkpoints/training_curves_v3.png')

print('Téléchargements lancés !')
print('-> Place waste_ai_v3.pt dans waste-ai/model/checkpoints/')
print('-> Met à jour api/main.py : remplace efficientnet_b2 et waste_ai_v3.pt')